In [1]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

import datetime

import pandas as pd

from pandas import ExcelWriter

from selenium import webdriver

from time import sleep

import os

from selenium.webdriver.common.by import By

from bs4 import BeautifulSoup

from selenium.webdriver.common.action_chains import ActionChains

import pdfplumber

from selenium.webdriver.support.ui import WebDriverWait

from selenium.webdriver.support import expected_conditions as EC

import tabula



In [2]:


# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'LI FMAL' ## change to current controller name



print(f"Running {regulatorName} Web Scraping Tool v.2.0")

now=datetime.datetime.now()

filename = '{} data {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)




Running LI FMAL Web Scraping Tool v.2.0


In [3]:

# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder,

         'profile.default_content_setting_values.automatic_downloads': 1 # Desable a Multiplefile download alert

         }

chromeOptions.add_argument("--disable-search-engine-choice-screen")

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()




In [4]:

# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict={

        # 'LI FMA 1': 'Bank',

        # 'LI FMA 2': 'Investment Firm',

        # 'LI FMA 3': 'E-Money Institution',

        # 'LI FMA 5': 'Asset Management Companies',

        # 'LI FMA 6': 'Bewilligte bzw. bescheinigte Investmentunternehmen (Fonds), AIF und OGAW' ,#' Approved or attested Investment Funds, AIF or UCITS', # --> PDF

        # 'LI FMA 7': 'Management Companies and AIFM', 

        # 'LI FMA 8': 'Fonds mit Vertriebszulassung im Fürstentum Liechtenstein' ,# information averifier,  # --> PDF 

        # 'LI FMA 9': 'Insurance Undertakings',

        # 'LI FMA 10': 'Liste der ausländischen Versicherungsunternehmen (EWR) (freier Dienstleistungsverkehr in Liechtenstein)' , # --> PDF

        # 'LI FMA 11': 'Liste der schweizerischen Versicherungsunternehmen (freier Dienstleistungs- oder Niederlassungsverkehr in Liechtenstein)', #  --> PDF

        # 'LI FMA 12': 'Occupation Pension schemes',

        # 'LI FMA 13': 'Pension Funds',

        # 'LI FMA 14': 'Professional Trustees',

         'LI FMA 15': '', # 'Mutationsliste x. Quartal 202x'  # --> PDF

        }



Typology={'LI FMA 1': 'Licensed credit institutions in Liechtenstein',

         'LI FMA 2': 'Licensed investment firms in Liechtenstein',

         'LI FMA 3': 'Electronic money institutions register',

         'LI FMA 4': 'EEA parent financial holding companies',

         'LI FMA 5': 'Licensed Asset Management Companies in Liechtenstein',

         'LI FMA 6': 'Approved or attested Investment Funds, AIF or UCITS',

         'LI FMA 7': 'Licensed Management Companies and AIFM in Liechtenstein',

         'LI FMA 8': 'ausländische Fonds mit Vertriebszulassung im Fürstentum Liechtenstein',

         'LI FMA 9': 'Versicherungsunternehmen',

         'LI FMA 10': 'Liste der ausländischen Versicherungsunternehmen (EWR), welche zum freien Dienstleistungsverkehr in Liechtenstein zugelassen sind',

         'LI FMA 11': 'Liste der schweizerischen Versicherungsunternehmen, welche zum freien Dienstleistungs- oder im Niederlassungsverkehr in Liechtenstein zugelassen sind',

         'LI FMA 12': 'Vorsorgeeinrichtungen im Fürstentum Liechtenstein',

         'LI FMA 13': 'Pensionsfonds im Fürstentum Liechtenstein',

         'LI FMA 14': 'Trustees with a licence for extensive activity',

         'LI FMA 15': 'Mutationslisten'

        }



sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

          'Phone - Mother company': [], 'Check': []}



contactData = ['Name', 'Address', 'Phone', 'E-Mail', 'Website', 'Registration number', 'Commercial Register No.']

processdate = now.strftime('%Y-%m-%d')




In [5]:

# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    len_value=[]

    for key, value in sqldict.items():

        len_value.append(len(value))

    maxlen = max(len_value)

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict



def click_element(driver, XPATH, Msg=None):

    for time in range(10):

        try:

            driver.find_element(By.XPATH, XPATH).click()

            if Msg!=None:

                print(f"{Msg}")

            break

        except:

            print(f"[ERROR] : - Retrying {time+1}/10 to click element in this page: {XPATH} ")

            sleep(1)

    else:

        raise Exception('[ERROR] : Failed to get presence for this element in this page:')



def check_dowload_files(tempfolder, fileType, wait_time=10):

    for time in range(wait_time):

        if len([ele for ele in os.listdir(tempfolder) if '.crdownload' not in ele and '.tmp' not in ele]) != 0 :

            print(f"[INFO] : {fileType} file = {os.listdir(tempfolder)})")

            break

        else:

            print(f"[INFO] : Download {fileType} file ... (wait {time*2}/20 s)")

            sleep(2)

    else:

        raise Exception(f'[ERROR] : Failed to Download {fileType} file. Run Script again' )

    return  os.listdir(tempfolder)[0]



def page_has_loaded(driver, timeout, POLL_FREQUENCY, XPATH, round=5):

    for time in range(round):

        try:

            element_present = EC.presence_of_element_located((By.XPATH, XPATH))

            #element_present = EC.element_to_be_clickable((By.XPATH, XPATH))

            wait=WebDriverWait(driver, timeout, POLL_FREQUENCY).until(element_present)

            break

        except:

            print(f"[ERROR] : Retrying {time+1}/10 to find element in this page: {XPATH} ")

            sleep(0.5)

    else:

        raise Exception('Failed to get presence for this element in this page:')

    return driver.find_element(By.XPATH, XPATH)



def compute_page(filePath, liste_expression1, liste_expression2=None ):

    Allpages = []

    AllIndexPages =[]

    with pdfplumber.open(filePath) as pdf:

        total_pages = len(pdf.pages)

        for i, expression1 in enumerate(liste_expression1) :

            pages = []

            IndexPages =[]

            for p, page in enumerate(pdf.pages):

                text = page.extract_text()

                if liste_expression2 != None :

                    if (expression1 in  text) or (liste_expression2[i] in  text) :

                        pages.append(p+1)

                        IndexPages.append(p+1)

                else:

                    if expression1 in  text :

                        pages.append(p+1)

                        IndexPages.append(p+1)

            Allpages.append(pages)

            AllIndexPages.append(IndexPages)

            pages = []

            IndexPages =[]

    return total_pages, Allpages, AllIndexPages



def Change_langage(lang):

    xpath_lang = '//a[@class="lang-nav"]'

    test = 1

    while driver.find_element(By.XPATH, xpath_lang).text == lang :

        click_element(driver,xpath_lang)

        test=+1

        if test==10:

            break

    sleep(1.5)




In [ ]:

# %%

#------------------------------------------------ Begin_Main ----------------------------------------

for k, reg in enumerate(regdict):



    print(f"[INFO] : Working {k + 1}/{len(regdict)} _({reg})_ ")



    if (reg == 'LI FMA 1') or (reg == 'LI FMA 2') or (reg == 'LI FMA 3') or (reg == 'LI FMA 5') or (reg == 'LI FMA 7') or (reg == 'LI FMA 9') or (reg == 'LI FMA 12') or (reg == 'LI FMA 13') or (reg == 'LI FMA 14'): 



        FMA_Reg_No = []

        ComRegNo = []

        Name = []

        Address = []

        RegulationType = []

        Type = []

        Granted = []

        #Registration_details =[]

        links = []

        driver.get('https://fmaregister.fma-li.li/search?lang=en')

        sleep(3)

        Change_langage('English')

        actionChains = ActionChains(driver)

        scrollingUp = driver.find_element(By.XPATH, '//*/label[contains(text(),"FMA Registration Number")]')

        driver.execute_script("arguments[0].scrollIntoView(true);", scrollingUp)

        sleep(0.1)

        click_element(driver, '//*/span[contains(text(),"bitte wählen")]') # click 'Category'

        sleep(0.2)

        # click_element(driver, f'//*/label[contains(text(),"{regdict[reg]}")]') # select 'Category'

        click_element(driver, f'//*/label[@title="{regdict[reg]}"]') # select 'Category' 

        click_element(driver, '//*/option[contains(text(),"Active entries")]') # Select 'Status'

        click_element(driver, '//*/input[@value="Search"]', '[INFO] : Click "Search" buttun') # click 'Search'

        sleep(3)



        soup = BeautifulSoup(driver.page_source, "html.parser")

        MultiPage = False

        total_pages=1

        pages =[]

        pagination = soup.find('nav', {'class': 'pagination pagination-bottom'})

        if pagination != None :

            if len(pagination)>0:

                for page in pagination.find_all('a'):

                    try:

                        num = int(page.text)

                        pages.append(num)

                    except:

                        continue

                total_pages=max(pages)

                MultiPage = True



        for page in range(total_pages):

            if MultiPage :

                click_element(driver, f'//nav[@class="pagination pagination-bottom"]//*/a[contains(text(),"{page+1}")]', f"[INFO] : - Scrapping page {page+1}/{total_pages} | {reg}")

                sleep(1)



            soup = BeautifulSoup(driver.page_source, "html.parser")

            results = soup.find('div', {'class':'form-register_result'}).find_all('div',{'class':'result-title'})

            result_detail = soup.find('div', {'class':'form-register_result'}).find_all('div',{'class':'result-detail'})



            for j, corporate in enumerate(results) :

                href = corporate.find('a', href=True)

                detailCorporate =  corporate.find_all('div')

                # Expired = result_detail[j].find('div', {'class':'result-info erloschen'}).text.strip()

                Type.append(result_detail[j].find('div', {'class':'result-info art'}).text)

                Granted.append(result_detail[j].find('div', {'class':'result-info registrierung'}).text)

                #Registration_details.append(result_detail[j].find('div', {'class': 'result-info auflagen'}).text)

                FMA_Reg_No.append(detailCorporate[0].text)

                ComRegNo.append(detailCorporate[1].text)

                Name.append(detailCorporate[2].text)

                Address.append(detailCorporate[3].text)

                links.append(None) if href is None else links.append(href['href'])

                # RegulationType.append('Licensed') if len(Expired) == 0 else RegulationType.append('License Lapsed')



                if reg == 'LI FMA 14' :

                    # RegulationType[len(RegulationType)-1] = result_detail[j].find('div', {'class': 'result-info auflagen'}).text

                    Granted[len(Granted) - 1] = ''





        for i, link in enumerate(links):

            print(f"[INFO] : -- webPage {i+1}/{len(links)} | {reg} ")

            valueLabel = ['']*len(contactData)



            if link is not None:

                driver.get('https://register.fma-li.li'+ link)

                sleep(1)

                page_has_loaded(driver, 5, 5, '/html/body/app-root/app-page-content')

                Change_langage('English')

                soup = BeautifulSoup(driver.page_source,'html.parser')

                trs= soup.find('tbody').find_all('tr')

                for tr in trs:

                    td = tr.find_all('td')

                    label = td[0].text

                    value = td[1].text

                    if label in contactData :

                        valueLabel[contactData.index(label)] = value



                Details_Address = valueLabel[1]

                sqldict['Phone'].append(valueLabel[2])

                sqldict['Email'].append(valueLabel[3])

                sqldict['Website'].append(valueLabel[4])



            if Address[i].strip()=='See Details':

                a_tags_with_specific_href = soup.find_all('a', href=lambda href: href and href.startswith(link))

                for tag in a_tags_with_specific_href:

                    if tag.text.strip() == regdict[reg]:

                        driver.get('https://register.fma-li.li'+ tag['href'])

                        sleep(1)

                        Change_langage('English')

                        soup = BeautifulSoup(driver.page_source,'html.parser')

                        trs= soup.find('tbody').find_all('tr')

                        for tr in trs:

                            td = tr.find_all('td')

                            label = td[0].text

                            value = td[1].text

                            if label in contactData :

                                valueLabel[contactData.index(label)] = value

                sqldict['Address_1'].append(valueLabel[1])

            else:

                sqldict['Address_1'].append(Address[i])



            sqldict['Name'].append(Name[i])

            if len(FMA_Reg_No[i]) > 0 :

                sqldict['InternalID_1_type'].append('FMA Registration number')

                sqldict['InternalID_1'].append(FMA_Reg_No[i])

            if len(ComRegNo[i]) > 0 :

                sqldict['InternalID_2_type'].append('Commercial Register Number')

                sqldict['InternalID_2'].append(ComRegNo[i])

            sqldict['RegulationType'].append('Regulated')

            sqldict['RegulationDate'].append(Granted[i])

            sqldict['ListProcessDate'].append(processdate)

            sqldict['RegCtry'].append(reg.split(' ')[0])

            sqldict['RegCode'].append(reg.split(' ')[1])

            sqldict['ListCode'].append(reg.split(' ')[-1])



            sqldict = bourange_same_length_array(sqldict)

        # click_element(driver, '//*/input[@value="Reset filters"]', '[INFO] : Click "Reset filters" buttun')






    else:

        driver.get('http://register.fma-li.li/')

        sleep(3)

        Change_langage('Deutsch')



        if (reg == 'LI FMA 15'):

            driver.get('https://www.fma-li.li/de/standort/finanzmarktteilnehmer')

            sleep(3)

            click_element(driver, '/html/body/dialog/div[2]/div/div[2]/div[2]/div[2]/div[3]') # click 'Category'

            mutations_h2 = page_has_loaded(
                driver,
                10,
                0.5,
                '//h2[contains(text(),"Mutationsliste")]',
                5
            )
            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", mutations_h2)
            sleep(1)
            version = [4,3,2,1]

            y = int(now.strftime('%Y'))

            yy = y-2

            stop = False

            while y != yy:

                for v in version :

                    pdf_label = f"Mutationsliste Q{v} {y}"

                    try:

                        pdf_xpath = (
                            f'//h2[contains(text(),"Mutationsliste")]'
                            f'/ancestor::div[contains(@class,"toolbox-element")]'
                            f'/following-sibling::div[contains(@class,"toolbox-download")][1]'
                            f'//a[contains(., "Mutationsliste Q{v} {y}") and contains(@href, ".pdf")]'
                        )

                        pdf_link = page_has_loaded(driver, 5, 0.5, pdf_xpath, 2)
                        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", pdf_link)
                        sleep(0.5)
                        pdf_link.click()

                        print(f"[INFO] : Downloaded PDF = {pdf_label}")

                        stop = True

                        break

                    except:

                        continue

                if stop:

                    break

                y-=1

        else:

            PDF_Register = page_has_loaded(driver, 5, 5, f'//*[@id="category"]/option[contains(text(),"{regdict[reg]}")]')

            PDF_Register.click()

            sleep(1)

            click_element(driver, '//*/input[@value="Anzeigen"]', '[INFO] : Click "Anzeigen" buttun') # Click 'Show'

        sleep(1)

        pdf_file = check_dowload_files(tempfolder, "PDF")

        filePath = os.path.join(tempfolder, pdf_file)



        # __Approved or attested Investment Funds, AIF or UCITS__(LI FMA 6)____

        if (reg == 'LI FMA 6'):

            total_pages, Allpages, AllIndexPages = compute_page(filePath, ['Fonds/OGAW/AIF']) #total_pages, Allpages, AllIndexPages = compute_page(filePath, ['Investment fund or Company'])

            tables = tabula.read_pdf(filePath, columns=[200, 360, 410], guess=False, pages=Allpages[0], encoding="cp1252")

            check_cell = ['Management_Company', 'Investment_fund']

            keyWords = ['AG', 'Ltd']

            for p, df in enumerate(tables):

                print(f"[INFO] : -- PdfPage {AllIndexPages[0][p]}/{total_pages} | {reg}")

                df.columns = ['Management_Company', 'Investment_fund', 'Type', 'Date_of_authorization']

                if p == 0:

                    date = df.iloc[4]['Management_Company'].split(":")[1].strip()

                    df = df[6:]

                    df = df[:-2]

                else:

                    df = df[1:]

                    df = df[:-2]

                df = df.reset_index(drop=True)

                for index, row in df.iterrows():

                    cell = str(row[check_cell[0]]).strip()

                    if (keyWords[0] in cell) or (keyWords[1] in cell):

                        if str(row[check_cell[1]]).strip() != 'nan':

                            name = str(row[check_cell[1]]).strip()

                            if index != 0:

                                preview_cell = str(df.iloc[index - 1][check_cell[0]]).strip()

                                preview_cell2 = str(df.iloc[index - 1][check_cell[1]]).strip()

                                if not (keyWords[0] in preview_cell) or not (keyWords[1] in preview_cell):

                                    name = preview_cell2 + " " + str(row[check_cell[1]]).strip()



                            if index < df.shape[0] - 1:

                                next_cell = str(df.iloc[index + 1][check_cell[0]]).strip()

                                next_cell2 = str(df.iloc[index + 1][check_cell[1]]).strip()

                                if not (keyWords[0] in next_cell) or not (keyWords[1] in next_cell):

                                    name = str(row[check_cell[1]]).strip() + " " + next_cell2

                        else:

                            name = str(df.iloc[index - 1][check_cell[1]]).strip()



                        # print(f"{name} | {date}")

                        sqldict['Name'].append(name)

                        # sqldict['Typology'].append(Typology[reg])

                        # sqldict['RegulationDate'].append(date)

                        sqldict['ListProcessDate'].append(processdate)

                        sqldict['RegCtry'].append(reg.split(' ')[0])

                        sqldict['RegCode'].append(reg.split(' ')[1])

                        sqldict['ListCode'].append(reg.split(' ')[-1])

                        sqldict['RegulationType'].append('Licensed')

                sqldict = bourange_same_length_array(sqldict)



        elif reg == 'LI FMA 8':

            count = 0

            total_pages, Allpages, AllIndexPages = compute_page(filePath, ['Teilfonds']) #total_pages, Allpages, AllIndexPages = compute_page(filePath, ['Investment fund or Company'])

            # tables_Umbrella_Fonds = tabula.read_pdf(filePath, columns=[205, 330, 485, 560, 660], guess=False, pages=Allpages[0])

            # date = ''

            # for p, df in enumerate(tables_Umbrella_Fonds):

            #     print(f"[INFO] : -- PdfPage {AllIndexPages[0][p]}/{total_pages} | {reg}")

            #     if df.shape[0] != 0:

            #         if p == 0:

            #             date = str(df.iloc[0:1, 0]).split()[2].replace('.', '-')

            #             df.columns = ['Fonds', 'Teilfonds', 'Verwaltungsgesellschaft', 'Herkunftsland', 'Zahlstelle','Vertreterstelle']

            #             df = df[2:]

            #             df = df.reset_index(drop=True)

            #         for index, row in df.iterrows():

            #             if str(row['Herkunftsland']) != 'nan':

            #                 count+=1

            #                 name = str(row['Fonds']).strip() + ' ' + str(row['Teilfonds']).strip()

            #                 if len(df)>=index+1 :

            #                     if str(df.loc[index+1, 'Herkunftsland']) == 'nan' and str(df.loc[index+1, 'Teilfonds']) != 'nan':

            #                         name = name + ' ' + df.loc[index+1, 'Teilfonds']


            #                 sqldict['Name'].append(name)

            #                 ##sqldict['RegulationDate'].append(date)

            #                 sqldict['Address_1'].append(row['Herkunftsland'])

            #                 sqldict['Cntry'].append(row['Herkunftsland'])

            #                 sqldict['RegulationType'].append('Supervised')

            #                 sqldict['ListProcessDate'].append(processdate)

            #                 sqldict['RegCtry'].append(reg.split(' ')[0])

            #                 sqldict['RegCode'].append(reg.split(' ')[1])

            #                 sqldict['ListCode'].append(reg.split(' ')[-1])


            #     sqldict = bourange_same_length_array(sqldict)



            page_Single_Fonds = list(set([i for i in range(1, total_pages+1)]) - set(Allpages[0]))

            tables_Single_Fonds = tabula.read_pdf(filePath, columns=[240, 480, 550, 650], guess=False, pages=page_Single_Fonds,encoding="cp1252")



            date = ''

            for p, df in enumerate(tables_Single_Fonds):

                print(f"[INFO] : -- PdfPage {page_Single_Fonds[p]}/{total_pages} | {reg}")

                if df.shape[0] != 0:

                    if p == 0:

                        date = str(df.iloc[0:1, 0]).split()[2].replace('.', '-')

                        df.columns = ['Fonds', 'Verwaltungsgesellschaft', 'Herkunftsland', 'Zahlstelle', 'Vertreterstelle']

                        df = df[2:-2]

                        df = df.reset_index(drop=True)

                    for index, row in df.iterrows():

                        if (str(row['Herkunftsland']) != 'nan') and (str(row['Fonds']) != 'nan'):

                            count+=1

                            name = str(row['Fonds']).strip()

                            if len(df)>=index+1 :

                                if str(df.loc[index+1, 'Herkunftsland']) == 'nan' and str(df.loc[index+1, 'Fonds']) != 'nan':

                                    name = name + ' ' + df.loc[index+1, 'Fonds']

                            sqldict['Name'].append(name)

                            # sqldict['RegulationDate'].append(date)

                            sqldict['Address_1'].append(row['Herkunftsland'])

                            sqldict['Cntry'].append(row['Herkunftsland'])

                            sqldict['RegulationType'].append('Regulated')

                            sqldict['ListProcessDate'].append(processdate)

                            sqldict['RegCtry'].append(reg.split(' ')[0])

                            sqldict['RegCode'].append(reg.split(' ')[1])

                            sqldict['ListCode'].append(reg.split(' ')[-1])

                

                sqldict = bourange_same_length_array(sqldict)

            

            print(f"[INFO] : ** count = {count}")



        elif reg == 'LI FMA 10':

            total_pages, Allpages, AllIndexPages = compute_page(filePath,['Versicherungszweig'])

            tables = tabula.read_pdf(filePath, columns=[140, 320, 460, 550, 670], guess=False, pages=Allpages[0])

            columns = ['Land', 'Name', 'Adresse', 'Ort', 'Leben', 'Nichtleben']

            for p, df in enumerate(tables):

                print(f"[INFO] : -- PdfPage {AllIndexPages[0][p]}/{total_pages} | {reg}")

                df.columns = columns

                if p == 0:

                    date = df.iloc[1]['Land'].split(":")[1].strip()

                    df = df[-1:]

                    df = df[:-1]

                    df = df.reset_index(drop=True)

                else:

                    df = df[2:]

                    df = df[:-1]

                    df = df.reset_index(drop=True)



                for index, row in df.iterrows():

                    # cell = str(row['Name']).strip()

                    name = str(row['Name']).strip()

                    ZipAndCity = str(row['Ort']).strip()

                    Adresse = str(row['Adresse']).strip()



                    extration_index = name[:3].strip()

                    extration_index = extration_index.split(' ')[0]

                    if str(extration_index).isdigit():

                        # name = cell

                        address = Adresse

                        if index + 1 < len(df):

                            if (str(df.iloc[index + 1]['Ort']) == "nan") and (str(df.iloc[index + 1]['Name']) != "nan"):

                                name = name + " " + str(df.iloc[index + 1]['Name'])

                            if (str(df.iloc[index + 1]['Name']) == "nan") and (str(df.iloc[index + 1]['Adresse']) != "nan"):

                                address = address + " " + str(df.iloc[index + 1]['Adresse'])



                        name =  name.replace(extration_index, '').strip()



                        ZipAndCity = str(row['Ort']).strip().split()

                        if len(ZipAndCity) > 1:

                            Zip = ZipAndCity[0]

                            City = ZipAndCity[1]

                        else:

                            Zip = ZipAndCity[0]



                        address = address + " " + Zip + " " + City

                        # print(f"{name} | {address} | {date}")

                        sqldict['Name'].append(name)

                        sqldict['Address_1'].append(address)

                        # sqldict['Typology'].append(Typology[reg])

                        sqldict['Zip'].append(Zip)

                        sqldict['City'].append(City)

                        # sqldict['RegulationDate'].append(date)

                        sqldict['ListProcessDate'].append(processdate)

                        sqldict['RegCtry'].append(reg.split(' ')[0])

                        sqldict['RegCode'].append(reg.split(' ')[1])

                        sqldict['ListCode'].append(reg.split(' ')[-1])

                        sqldict['RegulationType'].append('Licensed')



                sqldict = bourange_same_length_array(sqldict)



        # __Liste der schweizerischen Versicherungsunternehmen__(LI FMA 11)____

        elif reg == 'LI FMA 11':

            total_pages, Allpages, AllIndexPages = compute_page(filePath, ['Schadenversicherer'])

            tables = tabula.read_pdf(filePath, columns=[210,260, 350], guess=False, pages=Allpages[0])

            columns = ['Institut', 'Zulassungstyp', 'Zulassungstyp', 'xxx']

            for p, df in enumerate(tables):

                print(f"[INFO] : -- PdfPage {AllIndexPages[0][p]}/{total_pages} | {reg}")

                df.columns = columns

                if p == 0:

                    date = df.iloc[1]['Institut']

                    df = df[14:]

                    df = df[:-29]



                df.drop_duplicates(subset="Institut", keep='first', inplace=True)

                df = df.reset_index(drop=True)

                for index, row in df.iterrows():

                    cell = str(row['Institut']).strip()

                    # Zulassungstyp = str(row['Zulassungstyp']).strip()

                    # if ((Zulassungstyp.find('Krankenversicherer') != -1) or 

                    #     (Zulassungstyp.find('Schadenversicherer') != -1) or 

                    #     (Zulassungstyp.find('Lebensversicherer') != -1) or 

                    #     (Zulassungstyp.find('Krankenkasse') != -1)):



                    if (cell != 'nan'):

                        name = cell

                        # print(f"{name}")

                        sqldict['Name'].append(name)

                        # sqldict['Typology'].append(Typology[reg])

                        # sqldict['RegulationDate'].append(date)

                        sqldict['ListProcessDate'].append(processdate)

                        sqldict['RegCtry'].append(reg.split(' ')[0])

                        sqldict['RegCode'].append(reg.split(' ')[1])

                        sqldict['ListCode'].append(reg.split(' ')[-1])

                        sqldict['RegulationType'].append('Licensed')

                sqldict = bourange_same_length_array(sqldict)



        elif reg == 'LI FMA 15':

            total_pages, Allpages, AllIndexPages = compute_page(filePath, ['Bewilligungen', 'Investmentunternehmen'])

            Bewilligungen_pages = Allpages[0]

            Investmentunternehmen_pages = Allpages[1]

            # Neue_pages = Allpages[2]

            Bewilligungen_index_pages = AllIndexPages[0]

            Investmentunternehmen_index_pages = AllIndexPages[1]

            Neue_index_pages =AllIndexPages[0]

            tables_Bewilligungen = tabula.read_pdf(filePath, columns=[210, 370, 430], guess=False,

                                                   pages=Bewilligungen_pages)

            tables_Investmentunternehmen = tabula.read_pdf(filePath, columns=[200, 360, 430], guess=False,

                                                           pages=Investmentunternehmen_pages)

            # tables_Neue = tabula.read_pdf(filePath, columns=[210,410,590], guess=False, pages=Neue_pages)

            date = ''

            for p, df in enumerate(tables_Bewilligungen):

                print(f"[INFO] : -- PdfPage {Bewilligungen_index_pages[p]}/{total_pages} | {reg}")

                df.columns = ['Kategorie', 'Finanzintermediär', 'Datum', 'Bemerkungen']

                if df.shape[0] != 0:

                    if Bewilligungen_index_pages[p] == 1:

                        # RegulationType = 'Licensed'

                        date = df.iloc[1]['Kategorie'].split(":")[1].strip()

                        df = df[3:]

                        df = df[:-3]

                        df = df.reset_index(drop=True)

                    else:

                        # RegulationType = 'License Revoked'

                        df = df[:-2]

                        df = df.reset_index(drop=True)



                    for index, row in df.iterrows():

                        cell = str(row['Datum']).strip()

                        if (cell != 'nan'):



                            name = str(row['Finanzintermediär']).strip()

                            if index + 1 < len(df):

                                if (str(df.iloc[index + 1]['Datum']) == "nan") and (

                                        str(df.iloc[index + 1]['Finanzintermediär']) != "nan"):

                                    name = name + " " + str(df.iloc[index + 1]['Finanzintermediär'])



                            # print (f'{name}|{address} |{Zip} |{City}  | {date}')

                            sqldict['Name'].append(name)

                            # sqldict['Typology'].append(Typology[reg])

                            # sqldict['RegulationDate'].append(date)

                            sqldict['RegulationType'].append('Regulated')

                            sqldict['ListProcessDate'].append(processdate)

                            sqldict['RegCtry'].append(reg.split(' ')[0])

                            sqldict['RegCode'].append(reg.split(' ')[1])

                            sqldict['ListCode'].append(reg.split(' ')[-1])



                sqldict = bourange_same_length_array(sqldict)



            date = ''

            for p, df in enumerate(tables_Investmentunternehmen):

                print(f"[INFO] : -- PdfPage {Investmentunternehmen_index_pages[p]}/{total_pages} | {reg}")

                df.columns = ['Verwaltungsgesellschaft', 'Fonds', 'Fondstyp', 'Datum']

                if df.shape[0] != 0:
                    # if Investmentunternehmen_index_pages[p] == 2:

                        # RegulationType = 'Licensed'
                        try:

                            date = df.iloc[1]['Verwaltungsgesellschaft'].split(":")[1].strip()

                            df = df[3:]

                            df = df[:-5]

                            df = df.reset_index(drop=True)

                            # print(df)
                        except:
                            df = df[4:]

                            df = df[:-5]

                            df = df.reset_index(drop=True)
                            # print(df)


                for index, row in df.iterrows():

                    cell = str(row['Fondstyp']).strip()

                    if (cell == 'AIF') or (cell == 'OGAW V'):



                        name = str(row['Fonds']).strip()

                        if index + 1 < len(df):

                            if (str(df.iloc[index + 1]['Fondstyp']) == "nan") and (

                                    str(df.iloc[index + 1]['Fonds']) != "nan"):

                                name = name + " " + str(df.iloc[index + 1]['Fonds'])



                        # print (f'{name}|{address} |{Zip} |{City}  | {date}')

                        sqldict['Name'].append(name)

                        # sqldict['Typology'].append(Typology[reg])

                        # sqldict['RegulationDate'].append(date)

                        sqldict['RegulationType'].append('Regulated')

                        sqldict['ListProcessDate'].append(processdate)

                        sqldict['RegCtry'].append(reg.split(' ')[0])

                        sqldict['RegCode'].append(reg.split(' ')[1])

                        sqldict['ListCode'].append(reg.split(' ')[-1])



            sqldict = bourange_same_length_array(sqldict)



        for rem in os.listdir(tempfolder):

            os.remove(os.path.join(tempfolder, rem))



# %%

df



# %%

if len(df)>=3688 and str(df.loc[3688, 'Name']).find('Investors') != -1:

    print('OK')



[INFO] : Working 1/1 _(LI FMA 15)_ 
[ERROR] : Retrying 1/10 to find element in this page: //h2[contains(text(),"Mutationsliste")]/ancestor::div[contains(@class,"toolbox-element")]/following-sibling::div[contains(@class,"toolbox-download")][1]//a[contains(., "Mutationsliste Q4 2026") and contains(@href, ".pdf")] 
[ERROR] : Retrying 2/10 to find element in this page: //h2[contains(text(),"Mutationsliste")]/ancestor::div[contains(@class,"toolbox-element")]/following-sibling::div[contains(@class,"toolbox-download")][1]//a[contains(., "Mutationsliste Q4 2026") and contains(@href, ".pdf")] 
[ERROR] : Retrying 1/10 to find element in this page: //h2[contains(text(),"Mutationsliste")]/ancestor::div[contains(@class,"toolbox-element")]/following-sibling::div[contains(@class,"toolbox-download")][1]//a[contains(., "Mutationsliste Q3 2026") and contains(@href, ".pdf")] 
[ERROR] : Retrying 2/10 to find element in this page: //h2[contains(text(),"Mutationsliste")]/ancestor::div[contains(@class,"toolb

Failed to import jpype dependencies. Fallback to subprocess.
No module named 'jpype'


[INFO] : -- PdfPage 1/6 | LI FMA 15
[INFO] : -- PdfPage 4/6 | LI FMA 15
[INFO] : -- PdfPage 2/6 | LI FMA 15
        Verwaltungsgesellschaft                             Fonds Fondstyp  \
0                   Axxion S.A.                    ANKERCAP Funds      AIF   
1          LLB Fund Services AG           ASPOMA Japan Value Fund      AIF   
2      Accuro Fund Solutions AG               FlexInvest L/S Fund      AIF   
3      CAIAC Fund Management AG  Global Opportunities Funds SICAV      AIF   
4          LLB Fund Services AG               Hestia (Fund) SICAV      AIF   
5          IFM Independent Fund            India Capital Fund AIF      AIF   
6                 Management AG                               NaN      NaN   
7          IFM Independent Fund                  Investona Fund 2      AIF   
8                 Management AG                               NaN      NaN   
9          IFM Independent Fund                  Investona Fund 3      AIF   
10                Management AG   

In [7]:


# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)
    
    
    
    

C:\Users\wuj1\AppData\Local\Temp\3\ipykernel_39236\845782258.py:9: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'XlsxWriter' object has no attribute 'save'